<a href="https://colab.research.google.com/github/javisagredo-dev/Evaluacion2_Prog_DS/blob/main/04_aprendizaje_no_supervisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aprendizaje no supervisado: agrupamiento de variedades de frijol

## Importaciones necesarias

In [47]:
import pickle
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score, homogeneity_score, completeness_score, v_measure_score
import time
import warnings
warnings.filterwarnings('ignore')

## Carga y lectura de datos

In [48]:
df = pd.read_excel('Dry_Bean_Dataset.xlsx')

## Copia del dataset para aprendizaje no supervisado

In [49]:
df_model = df.copy()

## Separación de variables y codificación de etiquetas

In [50]:
X = df_model.drop('Class', axis=1)

le = LabelEncoder()
y_true = le.fit_transform(df_model['Class'])

print(f"Clases originales: {le.classes_}")
print(f"X shape: {X.shape}")

Clases originales: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']
X shape: (13611, 16)


## Estandarización de características con StandardScaler

In [51]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Escalado completado")
print(f"Media aproximada de X_scaled: {X_scaled.mean():.4f}")
print(f"Std aproximada de X_scaled: {X_scaled.std():.4f}")

Escalado completado
Media aproximada de X_scaled: -0.0000
Std aproximada de X_scaled: 1.0000


## K-Means: entrenamiento y evaluación

In [52]:
start_time = time.time()

kmeans = KMeans(n_clusters=7, init='random', n_init=3, max_iter=150, random_state=42)
labels_km = kmeans.fit_predict(X_scaled)

silhouette_km = silhouette_score(X_scaled, labels_km)
db_km = davies_bouldin_score(X_scaled, labels_km)
ari_km = adjusted_rand_score(y_true, labels_km)
homogeneity_km = homogeneity_score(y_true, labels_km)
completeness_km = completeness_score(y_true, labels_km)
vm_km = v_measure_score(y_true, labels_km)
time_km = time.time() - start_time

### Resultados

In [53]:
resultados = pd.DataFrame({
    'Métrica': ['Silhouette Score', 'Davies-Bouldin', 'Adjusted Rand Index', 'Homogeneidad', 'Completitud', 'V-Measure', 'Tiempo'],
    'Valor': [f"{silhouette_km:.4f}", f"{db_km:.4f}", f"{ari_km:.4f}", f"{homogeneity_km:.4f}", f"{completeness_km:.4f}", f"{vm_km:.4f}", f"{time_km:.2f}s"]
})
display(resultados)

,Métrica,Valor
0,Silhouette Score,0.3094
1,Davies-Bouldin,1.0993
2,Adjusted Rand Index,0.6686
3,Homogeneidad,0.7047
4,Completitud,0.7228
5,V-Measure,0.7137
6,Tiempo,1.73s


### Distribución de muestras por clúster

In [54]:
cluster_counts_km = pd.Series(labels_km).value_counts().sort_index()
dist_km = pd.DataFrame({
    'Clúster': cluster_counts_km.index,
    'Cantidad': cluster_counts_km.values,
    'Porcentaje': (cluster_counts_km.values / len(labels_km) * 100).round(2)
})
display(dist_km)

,Clúster,Cantidad,Porcentaje
0,0,540,3.97
1,1,1771,13.01
2,2,2033,14.94
3,3,3112,22.86
4,4,3155,23.18
5,5,520,3.82
6,6,2480,18.22


### Reporte por clúster

In [55]:
from sklearn.metrics import classification_report
print(classification_report(y_true, labels_km, labels=list(range(7)), target_names=[f"Cluster {i}" for i in range(7)]))

              precision    recall  f1-score   support

   Cluster 0       0.07      0.03      0.04      1322
   Cluster 1       0.00      0.00      0.00       522
   Cluster 2       0.00      0.00      0.00      1630
   Cluster 3       0.16      0.14      0.15      3546
   Cluster 4       0.00      0.00      0.00      1928
   Cluster 5       0.00      0.00      0.00      2027
   Cluster 6       0.00      0.00      0.00      2636

    accuracy                           0.04     13611
   macro avg       0.03      0.03      0.03     13611
weighted avg       0.05      0.04      0.04     13611



## Agrupamiento Jerárquico Aglomerativo: entrenamiento y evaluación

In [56]:
start_time = time.time()

agglo = AgglomerativeClustering(n_clusters=7, linkage='complete')
labels_ag = agglo.fit_predict(X_scaled)

silhouette_ag = silhouette_score(X_scaled, labels_ag)
db_ag = davies_bouldin_score(X_scaled, labels_ag)
ari_ag = adjusted_rand_score(y_true, labels_ag)
homogeneity_ag = homogeneity_score(y_true, labels_ag)
completeness_ag = completeness_score(y_true, labels_ag)
vm_ag = v_measure_score(y_true, labels_ag)
time_ag = time.time() - start_time

### Resultados

In [57]:
resultados = pd.DataFrame({
    'Métrica': ['Silhouette Score', 'Davies-Bouldin', 'Adjusted Rand Index', 'Homogeneidad', 'Completitud', 'V-Measure', 'Tiempo'],
    'Valor': [f"{silhouette_ag:.4f}", f"{db_ag:.4f}", f"{ari_ag:.4f}", f"{homogeneity_ag:.4f}", f"{completeness_ag:.4f}", f"{vm_ag:.4f}", f"{time_ag:.2f}s"]
})
display(resultados)

,Métrica,Valor
0,Silhouette Score,0.2282
1,Davies-Bouldin,1.1322
2,Adjusted Rand Index,0.3620
3,Homogeneidad,0.4366
4,Completitud,0.6438
5,V-Measure,0.5203
6,Tiempo,5.65s


### Distribución de muestras por clúster

In [58]:
cluster_counts_ag = pd.Series(labels_ag).value_counts().sort_index()
dist_ag = pd.DataFrame({
    'Clúster': cluster_counts_ag.index,
    'Cantidad': cluster_counts_ag.values,
    'Porcentaje': (cluster_counts_ag.values / len(labels_ag) * 100).round(2)
})
display(dist_ag)

,Clúster,Cantidad,Porcentaje
0,0,3935,28.91
1,1,494,3.63
2,2,5003,36.76
3,3,4096,30.09
4,4,52,0.38
5,5,29,0.21
6,6,2,0.01


### Reporte por clúster

In [59]:
print(classification_report(y_true, labels_ag, labels=list(range(7)), target_names=[f"Cluster {i}" for i in range(7)]))

              precision    recall  f1-score   support

   Cluster 0       0.18      0.55      0.27      1322
   Cluster 1       1.00      0.94      0.97       522
   Cluster 2       0.04      0.13      0.07      1630
   Cluster 3       0.50      0.58      0.54      3546
   Cluster 4       0.77      0.02      0.04      1928
   Cluster 5       0.00      0.00      0.00      2027
   Cluster 6       0.00      0.00      0.00      2636

    accuracy                           0.26     13611
   macro avg       0.36      0.32      0.27     13611
weighted avg       0.30      0.26      0.22     13611



## DBSCAN: entrenamiento y evaluación

In [60]:
start_time = time.time()

dbscan = DBSCAN(eps=0.6, min_samples=15)
labels_db = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = list(labels_db).count(-1)

mask_db = labels_db != -1
silhouette_db = silhouette_score(X_scaled[mask_db], labels_db[mask_db]) if n_clusters_db >= 2 else float('nan')
db_db = davies_bouldin_score(X_scaled[mask_db], labels_db[mask_db]) if n_clusters_db >= 2 else float('nan')
ari_db = adjusted_rand_score(y_true[mask_db], labels_db[mask_db]) if n_clusters_db >= 2 else float('nan')
homogeneity_db = homogeneity_score(y_true[mask_db], labels_db[mask_db]) if n_clusters_db >= 2 else float('nan')
completeness_db = completeness_score(y_true[mask_db], labels_db[mask_db]) if n_clusters_db >= 2 else float('nan')
vm_db = v_measure_score(y_true[mask_db], labels_db[mask_db]) if n_clusters_db >= 2 else float('nan')
time_db = time.time() - start_time

print(f"Clústeres encontrados: {n_clusters_db}")
print(f"Puntos de ruido: {n_noise_db} ({n_noise_db/len(labels_db)*100:.2f}%)")

Clústeres encontrados: 11
Puntos de ruido: 6164 (45.29%)


### Resultados

In [61]:
resultados = pd.DataFrame({
    'Métrica': ['Clústeres encontrados', 'Puntos de ruido', 'Silhouette Score', 'Davies-Bouldin', 'Adjusted Rand Index', 'Homogeneidad', 'Completitud', 'V-Measure', 'Tiempo'],
    'Valor': [str(n_clusters_db), str(n_noise_db), f"{silhouette_db:.4f}", f"{db_db:.4f}", f"{ari_db:.4f}", f"{homogeneity_db:.4f}", f"{completeness_db:.4f}", f"{vm_db:.4f}", f"{time_db:.2f}s"]
})
display(resultados)

,Métrica,Valor
0,Clústeres encontrados,11
1,Puntos de ruido,6164
2,Silhouette Score,0.1558
3,Davies-Bouldin,1.0952
4,Adjusted Rand Index,0.1314
5,Homogeneidad,0.2364
6,Completitud,0.7448
7,V-Measure,0.3589
8,Tiempo,0.89s


### Distribución de muestras por clúster

In [62]:
cluster_counts_db = pd.Series(labels_db).value_counts().sort_index()
dist_db = pd.DataFrame({
    'Clúster': cluster_counts_db.index,
    'Cantidad': cluster_counts_db.values,
    'Porcentaje': (cluster_counts_db.values / len(labels_db) * 100).round(2)
})
display(dist_db)

,Clúster,Cantidad,Porcentaje
0,-1,6164,45.29
1,0,6851,50.33
2,1,164,1.20
3,2,41,0.30
4,3,13,0.10
5,4,11,0.08
6,5,19,0.14
7,6,177,1.30
8,7,51,0.37
9,8,99,0.73


### Reporte por clúster

In [63]:
labels_db_reindexed = labels_db[mask_db]
unique_clusters = sorted(set(labels_db_reindexed))
print(classification_report(y_true[mask_db], labels_db_reindexed, labels=unique_clusters, target_names=[f"Cluster {i}" for i in unique_clusters]))

              precision    recall  f1-score   support

   Cluster 0       0.00      0.73      0.00        15
   Cluster 1       0.00      0.00      0.00         0
   Cluster 2       0.98      0.17      0.30       230
   Cluster 3       0.00      0.00      0.00      2953
   Cluster 4       0.00      0.00      0.00       376
   Cluster 5       0.00      0.00      0.00      1797
   Cluster 6       0.00      0.00      0.00      2076
   Cluster 7       0.00      0.00      0.00         0
   Cluster 8       0.00      0.00      0.00         0
   Cluster 9       0.00      0.00      0.00         0
  Cluster 10       0.00      0.00      0.00         0

    accuracy                           0.01      7447
   macro avg       0.09      0.08      0.03      7447
weighted avg       0.03      0.01      0.01      7447



## Comparación de modelos

In [64]:
comparison = pd.DataFrame({
    'Modelo': ['K-Means', 'Jerárquico (Complete)', 'DBSCAN'],
    'Silhouette Score': [silhouette_km, silhouette_ag, silhouette_db],
    'Davies-Bouldin': [db_km, db_ag, db_db],
    'ARI': [ari_km, ari_ag, ari_db],
    'V-Measure': [vm_km, vm_ag, vm_db],
    'Tiempo (s)': [time_km, time_ag, time_db]
})

display(comparison.round(4))

best_model = comparison.loc[comparison['ARI'].idxmax(), 'Modelo']
best_ari = comparison['ARI'].max()
best_sil = comparison.loc[comparison['ARI'].idxmax(), 'Silhouette Score']

print(f"\n{'='*60}")
print(f"MEJOR MODELO: {best_model}")
print(f"ARI: {best_ari:.4f}")
print(f"Silhouette Score: {best_sil:.4f}")
print(f"{'='*60}")

,Modelo,Silhouette Score,Davies-Bouldin,ARI,V-Measure,Tiempo (s)
0,K-Means,0.3094,1.0993,0.6686,0.7137,1.7275
1,Jerárquico (Complete),0.2282,1.1322,0.3620,0.5203,5.6488
2,DBSCAN,0.1558,1.0952,0.1314,0.3589,0.8904



MEJOR MODELO: K-Means
ARI: 0.6686
Silhouette Score: 0.3094


## Verificación de estabilidad de clústeres

In [65]:
semillas = [0, 1, 42, 99, 123]

aris_km = []
sils_km = []
for seed in semillas:
    km_temp = KMeans(n_clusters=7, init='random', n_init=3, max_iter=150, random_state=seed)
    lbl = km_temp.fit_predict(X_scaled)
    aris_km.append(adjusted_rand_score(y_true, lbl))
    sils_km.append(silhouette_score(X_scaled, lbl))

aris_ag = []
sils_ag = []
for seed in semillas:
    ag_temp = AgglomerativeClustering(n_clusters=7, linkage='complete')
    lbl = ag_temp.fit_predict(X_scaled)
    aris_ag.append(adjusted_rand_score(y_true, lbl))
    sils_ag.append(silhouette_score(X_scaled, lbl))

stability_df = pd.DataFrame({
    'Modelo': [' K-Means', ' Jerárquico (Complete)'],
    'ARI Media': [np.mean(aris_km), np.mean(aris_ag)],
    'ARI Std': [np.std(aris_km), np.std(aris_ag)],
    'Silhouette Media': [np.mean(sils_km), np.mean(sils_ag)],
    'Silhouette Std': [np.std(sils_km), np.std(sils_ag)]
})

display(stability_df.round(4))

from IPython.display import Markdown
display(Markdown("###  Interpretación"))

for i, row in stability_df.iterrows():
    std = row['ARI Std']
    if std < 0.01:
        estado = "Excelente (muy estable)"
    elif std < 0.03:
        estado = "Aceptable"
    else:
        estado = "Inestable"
    print(f"**{row['Modelo']}:** ARI Std = {std:.4f} → {estado}")

display(Markdown("---"))
display(Markdown("**Referencia:**"))
display(Markdown("- ARI Std < 0.01 → Excelente (muy estable)"))
display(Markdown("- ARI Std 0.01-0.03 → Aceptable"))
display(Markdown("- ARI Std > 0.03 → Inestable"))

best_stable = stability_df.loc[stability_df['ARI Std'].idxmin(), 'Modelo']
print(f"\n Modelo más estable: {best_stable}")

,Modelo,ARI Media,ARI Std,Silhouette Media,Silhouette Std
0,K-Means,0.651,0.0354,0.2985,0.0217
1,Jerárquico (Complete),0.362,0.0000,0.2282,0.0000


###  Interpretación

** K-Means:** ARI Std = 0.0354 → Inestable
** Jerárquico (Complete):** ARI Std = 0.0000 → Excelente (muy estable)


---

**Referencia:**

- ARI Std < 0.01 → Excelente (muy estable)

- ARI Std 0.01-0.03 → Aceptable

- ARI Std > 0.03 → Inestable


 Modelo más estable:  Jerárquico (Complete)


## Guardado de resultados para importar en optimización

In [66]:
variables_guardar = {
    # ARI
    'ari_km': ari_km,
    'ari_ag': ari_ag,
    'ari_db': ari_db,

    # Silhouette
    'silhouette_km': silhouette_km,
    'silhouette_ag': silhouette_ag,
    'silhouette_db': silhouette_db,

    # Davies-Bouldin
    'db_km': db_km,
    'db_ag': db_ag,
    'db_db': db_db,

    # V-Measure
    'vm_km': vm_km,
    'vm_ag': vm_ag,
    'vm_db': vm_db,

    # Etiquetas
    'labels_km': labels_km,
    'labels_ag': labels_ag,
    'labels_db': labels_db,

    # Otros
    'y_true': y_true,
    'le': le
}

with open('resultados_no_supervisado.pkl', 'wb') as f:
    pickle.dump(variables_guardar, f)